In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import KFold
from scipy.optimize import minimize

In [ ]:
# --- Load Data ---
try:
    train_df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test.csv")
except FileNotFoundError:
    print("Ensure train.csv and test.csv are in the same directory.")
    exit()

In [ ]:
# --- Advanced Feature Engineering ---
def create_features(df):
    for i in range(1, 11):
        df[f'weighted_property_{i}'] = sum(df[f'Component{j}_fraction'] * df[f'Component{j}_Property{i}'] for j in range(1, 6))

    fractions = [f'Component{j}_fraction' for j in range(1, 6)]
    for i in range(len(fractions)):
        for j in range(i + 1, len(fractions)):
            df[f'frac_interaction_{i+1}_{j+1}'] = df[fractions[i]] * df[fractions[j]]

    for i in range(1, 11):
        prop_cols = [f'Component{j}_Property{i}' for j in range(1, 6)]
        df[f'prop_mean_{i}'] = df[prop_cols].mean(axis=1)
        df[f'prop_std_{i}'] = df[prop_cols].std(axis=1)
        for j in range(1, 6):
            df[f'Component{j}_prop{i}_diff_from_mean'] = df[f'Component{j}_Property{i}'] - df[f'prop_mean_{i}']

    return df

print("Creating advanced features...")
train_df = create_features(train_df)
test_df = create_features(test_df)

In [ ]:
# --- Define MAPE and Model Parameters ---
features = [col for col in train_df.columns if col not in ['ID'] + [f'BlendProperty{i}' for i in range(1, 11)]]
targets = [f'BlendProperty{i}' for i in range(1, 11)]

X_train = train_df[features]
y_train = train_df[targets]
X_test = test_df[features]

def safe_mape(y_true, y_pred):
    epsilon = 1e-9
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

lgbm_params = {'objective': 'mape', 'metric': 'mape', 'n_estimators': 2000, 'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1, 'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': 42}
xgb_params = {'objective': 'reg:squarederror', 'n_estimators': 2000, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.8, 'max_depth': 6, 'verbosity': 0, 'n_jobs': -1, 'seed': 42}
cat_params = {'iterations': 2000, 'learning_rate': 0.02, 'loss_function': 'MAPE', 'eval_metric': 'MAPE', 'depth': 6, 'l2_leaf_reg': 3, 'verbose': 0, 'random_seed': 42}

In [ ]:
# --- K-Fold Training with Weighted Ensembling ---
NFOLDS = 10
folds = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)
final_predictions = pd.DataFrame({'ID': test_df['ID']})

def get_optimal_weights(y_true, oof_preds):
    def loss_func(weights):
        weights = np.abs(weights) / np.sum(np.abs(weights))
        final_pred = sum(w * p for w, p in zip(weights, oof_preds))
        return safe_mape(y_true, final_pred)

    starting_weights = [1/len(oof_preds)] * len(oof_preds)
    bounds = [(0, 1) for _ in range(len(oof_preds))]
    constraints = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})

    res = minimize(loss_func, starting_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x

for target in targets:
    print(f"🎯 Training and finding weights for {target}...")

    lgb_preds, xgb_preds, cat_preds = np.zeros(len(X_test)), np.zeros(len(X_test)), np.zeros(len(X_test))
    oof_lgb, oof_xgb, oof_cat = np.zeros(len(X_train)), np.zeros(len(X_train)), np.zeros(len(X_train))

    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train[target])):
        print(f"  - Fold {fold_ + 1}/{NFOLDS}")
        X_trn, y_trn = X_train.iloc[trn_idx], y_train[target].iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train[target].iloc[val_idx]

        # Train LightGBM and CatBoost with early stopping
        lgb_model = lgb.LGBMRegressor(**lgbm_params); lgb_model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
        cat_model = cb.CatBoostRegressor(**cat_params); cat_model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], early_stopping_rounds=100, use_best_model=True, verbose=0)

        # ** FIX: Train XGBoost without early stopping to ensure compatibility **
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_trn, y_trn) # No eval_set, no early stopping

        # Store Predictions
        oof_lgb[val_idx], oof_xgb[val_idx], oof_cat[val_idx] = lgb_model.predict(X_val), xgb_model.predict(X_val), cat_model.predict(X_val)
        lgb_preds += lgb_model.predict(X_test) / NFOLDS
        xgb_preds += xgb_model.predict(X_test) / NFOLDS
        cat_preds += cat_model.predict(X_test) / NFOLDS

    # Find and apply optimal weights
    optimal_weights = get_optimal_weights(y_train[target], [oof_lgb, oof_xgb, oof_cat])
    print(f"  Optimal weights for {target}: LGB={optimal_weights[0]:.4f}, XGB={optimal_weights[1]:.4f}, CAT={optimal_weights[2]:.4f}")

    final_target_preds = (optimal_weights[0] * lgb_preds + optimal_weights[1] * xgb_preds + optimal_weights[2] * cat_preds)
    final_predictions[target] = final_target_preds

In [ ]:
# --- Create Submission ---
final_predictions.to_csv('submission_mape_ensemble_compatible.csv', index=False)

print("\n Submission file 'submission_mape_ensemble_compatible.csv' has been created successfully!")
print(final_predictions.head())